# Getting Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from multiprocessing import Process
import multiprocessing
import time

In [ ]:
pkt_prob = 0.5
e_prob = 0.5
weight_prob = 0.5

In [ ]:
h_vals_u1 = [0.1, 1]
h_bad = 0.5
h1_prob = [h_bad, (1-h_bad)]
weight_vals = [1, 2]
B_max = 2
rmax = 1
B1 = np.arange(0, B_max+1, 1)
rem_bits_1 = np.arange(0, rmax+0.5, 0.5)
all_states = []
for i in B1:
    for i2 in B1:
        for k in rem_bits_1:
            for k2 in rem_bits_1:
                for m in h_vals_u1:
                    for m2 in h_vals_u1:
                        for n in weight_vals:
                            for n2 in weight_vals:
                                all_states.append((i, i2, k, k2, m,m2, n, n2))
all_states = np.array(all_states)
print(all_states.shape)    

In [ ]:
P1 = np.arange(0, B_max+1, 1)
rho_1 =np.arange(0, rmax+0.5, 0.5)
all_actions =[]
for i in P1:
    for i2 in P1:
        for k in rho_1:
            for k2 in  rho_1:
                all_actions.append((i,i2, k,k2))
all_actions = np.array(all_actions)
print(all_actions.shape)

In [ ]:
h = 1

P1 = 2
P2 = 2

np.log2(1 + h*P1 + P2)

In [ ]:
def st_tr_B(B, B_n, P):
    if B - P < 0:
        B_ch = B
    else:
        B_ch = B - P
    if (B_ch == B_max and B_n == B_max):
        m = 1
    else:
        if(B_ch == B_n):
            m = 1 - e_prob
        elif (B_ch+1 == B_n):
            m = e_prob
        else:
            m = 0
    return m
def st_tr_rem (rem_b, rem_b_next, rho):
    # rho = float(rho)
    if (rem_b - rho) < 0:
        rem_ch = rem_b
    else:
        rem_ch = rem_b - rho

    if (rem_ch == rmax and rem_b_next == rmax):
        m = 1
    else:
        if rem_ch == rem_b_next:
            m = 1 - pkt_prob
        elif (rem_b_next == rmax):
            m = pkt_prob
        else:
            m = 0
    return m
def st_tr_wt(wt, wt_next):
    if wt == 1 and wt_next == 1:
        m = (1-pkt_prob) + (pkt_prob) * (weight_prob)
    elif wt == 1 and wt_next == 2:
        m = pkt_prob * (1 - weight_prob)
    elif wt == 2 and wt_next == 1:
        m = pkt_prob * weight_prob
    else:
        m = (1-pkt_prob) + pkt_prob * (1-weight_prob)
    return m

In [ ]:
def state_trans_prob(state, next_state, action):
    st_tr_prob_B1 = st_tr_B(state[0], next_state[0], action[0])
    st_tr_prob_B2 = st_tr_B(state[1], next_state[1], action[1])
    st_tr_prob_rem1 = st_tr_rem(state[2], next_state[2] , action[2])
    st_tr_prob_rem2 = st_tr_rem(state[3], next_state[3] , action[3])
    st_tr_weight1 = st_tr_wt(state[6], next_state[6])
    st_tr_weight2 = st_tr_wt(state[7], next_state[7])
    idx1 = h_vals_u1.index(next_state[4])
    idx2 = h_vals_u1.index(next_state[5])
    st_tr_h1 = h1_prob[idx1]
    st_tr_h2 = h1_prob[idx2]
    return st_tr_prob_B1 * st_tr_prob_B2 * st_tr_prob_rem1  * st_tr_prob_rem2 * \
            st_tr_h1 * st_tr_h2 *st_tr_weight1 * st_tr_weight2

def reward_fn(state, action):
    B1, B2, rem1, rem2, h1, h2, wt1, wt2 = state[0], state[1], state[2], state[3], state[4], state[5], state[6], state[7]
    P1, P2, rho1, rho2 = action[0], action[1], action[2], action[3]
    cost = 0
    m1 = np.log2(1 + h1* P1)
    m2  = np.log2(1 + h2 * P2)
    m3 = np.log2( 1 + h1*P1 + h2*P2)
    rem_end1 = rem1 - rho1
    rem_end2 = rem2 - rho2
    if (P1 >  0 and rho1  ==  0):
        cost = 10000
    elif (P2 > 0  and rho2 ==0):
        cost = 10000
    elif (P1 > B1):
        cost = 10000
    elif(P2 > B2):
        cost = 10000
    elif(rho2 > rem2):
        cost = 10000
    elif (rho1 > rem1):
        cost = 10000
    elif (rho1 > m1 ):
        cost = 10000
    elif (rho2 > m2):
        cost  = 10000
    elif (rho1+rho2 > m3):
        cost  = 10000
    else:
        cost = wt1 * np.exp(-(rmax - rem_end1)) + wt2 * np.exp(-(rmax - rem_end2))
    return cost

In [ ]:
def value_iteration(j, V_T, send_end):
    act_arr = []
    for act in range(len(all_actions)):
        R_sum = reward_fn(all_states[j], all_actions[act])
        mul1 = 0
        for itr in range(ls):
            v = V_T[itr]
            P_s_tr = state_trans_prob(all_states[j], all_states[itr], all_actions[act])
            mul1 = mul1 + (v*P_s_tr)
        act_arr.append(gamma * mul1 + R_sum)
    fin_v = np.min(act_arr)
    idx = act_arr.index(fin_v)
    temp = [fin_v, idx]
    send_end.send(temp)

In [ ]:
T = 5
ls = len(all_states)
V_T = np.zeros(ls)
Ix_T = np.zeros(ls)
gamma = 0.99
stp = 0
while(stp < T):
    t1 = time.time()
    print(stp)
    q = 0
    z = 36
    V_Ttemp = np.array([])
    Ix_Ttemp = np.array([])
    while z <= len(all_states):
        jobs = []
        pipe_list = []
        for j in range(q, z):
            recv_end, send_end = multiprocessing.Pipe(False)
            p = Process(target=value_iteration, args = (j, V_T, send_end))
            jobs.append(p)
            pipe_list.append(recv_end)
        for process in jobs:
            process.start()
        for process in jobs:
            process.join()
        temp2 = np.array([x.recv() for x in pipe_list])
        V_Ttemp = np.concatenate((V_Ttemp, temp2[:, 0]))
        Ix_Ttemp = np.concatenate((Ix_Ttemp, temp2[:, 1]))
        q = z
        z+= 36
    V_T = V_Ttemp
    print('time per itr', time.time() - t1)
    stp+= 1
print('value iteration done', e_prob, pkt_prob, weight_prob)

In [ ]:
Ix_T = Ix_Ttemp

In [ ]:
for i in range(len(all_states)):
    idx = int(Ix_T[i])
    act = all_actions[idx]
    print(all_states[i],'--',act)

In [ ]:
M = 2
T_horizon = 1000
tot_dist = 0
state_slot = [0, 0, 0, 0, 0.1, 0.1,  1, 1]
wt_next1, wt_next2 = 1, 1
P1, P2, rho1, rho2 = 0, 0, 0, 0
for t in range(T_horizon):

    wt1 = np.random.choice([1, 2], p = [weight_prob, (1-weight_prob)])
    wt2 = np.random.choice([1, 2], p = [weight_prob, (1-weight_prob)])
    h1 = np.random.choice([0.1, 1], p = h1_prob)
    h2 = np.random.choice([0.1, 1], p = h1_prob)
    pkt1 = np.random.choice([1, 0], p=[pkt_prob, (1 - pkt_prob)])
    pkt2 = np.random.choice([1, 0], p=[pkt_prob, (1 - pkt_prob)])
    E1 = np.random.choice([1, 0], p=[ e_prob, (1-e_prob)])
    E2 = np.random.choice([1, 0], p=[ e_prob, (1-e_prob)])

    ###############################################################
    # if pkt1 == 1:
    #     next_bits1 = rmax
    #     wt_next1 = wt1
    # else:
    #     next_bits1 = state_slot[2]

    # if pkt2 == 1:
    #     next_bits2 = rmax
    #     wt_next2 = wt2
    # else:
    #     next_bits2 = state_slot[3]
    ###############################################################
    # if pkt1 == 1:
    #     if wt1 >= wt_next1:
    #         next_bits1 = rmax
    #         wt_next1 = wt1
    #     else:
    #         next_bits1 = state_slot[2] 
    # else:
    #     next_bits1 = state_slot[2] 

    # if pkt2 == 1:
    #     if wt2 >= wt_next2:
    #         next_bits2 = rmax
    #         wt_next2 = wt2
    #     else:
    #         state_slot[3]
    # else:
    #     next_bits2 = state_slot[3] 
    ########################################################################
    # if pkt1 == 1:

    #     tempw1a = wt_next1 * np.exp(-(rmax - (state_slot[2]) ))
    #     tempw2a = wt1 * 1

    #     if tempw2a > tempw1a:
    #         next_bits1 = rmax
    #         wt_next1 = wt1
    #     else:
    #         next_bits1 = state_slot[2]
            
    # else:
    #     next_bits1 = state_slot[2]

    # if pkt2 == 1:

    #     tempw1b = wt_next2 * np.exp(-(rmax - (state_slot[3]) ))
    #     tempw2b = wt2 * 1

    #     if tempw2b > tempw1b:
    #         next_bits2 = rmax
    #         wt_next2 = wt2
    #     else:
    #         state_slot[3]
    # else:
    #     next_bits2 = state_slot[3] 

    print(wt_next1, wt_next2, next_bits1, next_bits2)

    # #####################################################################################

    next_B1 = np.minimum(state_slot[0]+ E1, B_max)
    next_B2 = np.minimum(state_slot[1]+ E2, B_max)

    state_slot = [next_B1, next_B2, next_bits1, next_bits2, h1, h2, wt_next1, wt_next2]

    s_idx = all_states.tolist().index(state_slot)
    idx = int(Ix_T[s_idx])
    act = all_actions[idx]
    P1, P2,rho1, rho2 = act[0], act[1], act[2], act[3]
    ################################################################
    dist = wt_next1 * np.exp(-(rmax - (state_slot[2]-rho1) )) + wt_next2 * np.exp(-(rmax - (state_slot[3]-rho2) ))
    state_slot = [state_slot[0]-P1, state_slot[1]-P2, state_slot[2] - rho1, state_slot[3] - rho2, state_slot[4], state_slot[5], state_slot[6], state_slot[7]]
    ########################################################################
    tot_dist+=  dist
fin_result = tot_dist/(T_horizon*M)
print('final objective', fin_result)
print('-------------------------------------')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from multiprocessing import Process
import multiprocessing
import time

# Final

In [ ]:
def MDP_two_user(pkt_prob, e_prob, weight_prob, T_horizon, send_end):
    h_vals_u1 = [0.1, 1]
    h_bad = 0.5
    h1_prob = [h_bad, (1-h_bad)]
    weight_vals = [1, 2]
    B_max = 2
    rmax = 1
    B1 = np.arange(0, B_max+1, 1)
    rem_bits_1 = np.arange(0, rmax+0.5, 0.5)
    all_states = []
    for i in B1:
        for i2 in B1:
            for k in rem_bits_1:
                for k2 in rem_bits_1:
                    for m in h_vals_u1:
                        for m2 in h_vals_u1:
                            for n in weight_vals:
                                for n2 in weight_vals:
                                    all_states.append((i, i2, k, k2, m,m2, n, n2))
    all_states = np.array(all_states)
    P1 = np.arange(0, B_max+1, 1)
    rho_1 =np.arange(0, rmax+0.5, 0.5)
    all_actions =[]
    for i in P1:
        for i2 in P1:
            for k in rho_1:
                for k2 in  rho_1:
                    all_actions.append((i,i2, k,k2))
    all_actions = np.array(all_actions)
    def st_tr_B(B, B_n, P):
        if B - P < 0:
            B_ch = B
        else:
            B_ch = B - P

        if (B_ch == B_max and B_n == B_max):
            m = 1
        else:
            if(B_ch == B_n):
                m = 1 - e_prob
            elif (B_ch+1 == B_n):
                m = e_prob
            else:
                m = 0
        return m
    def st_tr_rem (rem_b, rem_b_next, rho):
        # rho = float(rho)
        if (rem_b - rho) < 0:
            rem_ch = rem_b
        else:
            rem_ch = rem_b - rho

        if (rem_ch == rmax and rem_b_next == rmax):
            m = 1
        else:
            if rem_ch == rem_b_next:
                m = 1 - pkt_prob
            elif (rem_b_next == rmax):
                m = pkt_prob
            else:
                m = 0
        return m
    def st_tr_wt(wt, wt_next):
        if wt == 1 and wt_next == 1:
            m = (1-pkt_prob) + (pkt_prob) * (weight_prob)
        elif wt == 1 and wt_next == 2:
            m = pkt_prob * (1 - weight_prob)
        elif wt == 2 and wt_next == 1:
            m = pkt_prob * weight_prob
        else:
            m = (1-pkt_prob) + pkt_prob * (1-weight_prob)
        return m
        

    def state_trans_prob(state, next_state, action):
        st_tr_prob_B1 = st_tr_B(state[0], next_state[0], action[0])
        st_tr_prob_B2 = st_tr_B(state[1], next_state[1], action[1])
        st_tr_prob_rem1 = st_tr_rem(state[2], next_state[2] , action[2])
        st_tr_prob_rem2 = st_tr_rem(state[3], next_state[3] , action[3])
        st_tr_weight1 = st_tr_wt(state[6], next_state[6])
        st_tr_weight2 = st_tr_wt(state[7], next_state[7])
        idx1 = h_vals_u1.index(next_state[4])
        idx2 = h_vals_u1.index(next_state[5])
        st_tr_h1 = h1_prob[idx1]
        st_tr_h2 = h1_prob[idx2]
        return st_tr_prob_B1 * st_tr_prob_B2 * st_tr_prob_rem1  * st_tr_prob_rem2 * \
                st_tr_h1 * st_tr_h2 *st_tr_weight1 * st_tr_weight2

    def reward_fn(state, action):
        B1, B2, rem1, rem2, h1, h2, wt1, wt2 = state[0], state[1], state[2], state[3], state[4], state[5], state[6], state[7]
        P1, P2, rho1, rho2 = action[0], action[1], action[2], action[3]
        cost = 0
        m1 = np.log2(1 + h1* P1)
        m2  = np.log2(1 + h2 * P2)
        m3 = np.log2( 1 + h1*P1 + h2*P2)
        rem_end1 = rem1 - rho1
        rem_end2 = rem2 - rho2
        if (P1 >  0 and rho1  ==  0):
            cost = 10000
        elif (P2 > 0  and rho2 ==0):
            cost = 10000
        elif (P1 > B1):
            cost = 10000
        elif(P2 > B2):
            cost = 10000
        elif(rho2 > rem2):
            cost = 10000
        elif (rho1 > rem1):
            cost = 10000
        elif (rho1 > m1 ):
            cost = 10000
        elif (rho2 > m2):
            cost  = 10000
        elif (rho1+rho2 > m3):
            cost  = 10000
        else:
            cost = wt1 * np.exp(-(rmax - rem_end1)) + wt2 * np.exp(-(rmax - rem_end2))
        return cost

    def value_iteration(j, V_T, send_end):
        act_arr = []
        for act in range(len(all_actions)):
            R_sum = reward_fn(all_states[j], all_actions[act])
            mul1 = 0
            for itr in range(ls):
                v = V_T[itr]
                P_s_tr = state_trans_prob(all_states[j], all_states[itr], all_actions[act])
                mul1 = mul1 + (v*P_s_tr)
            act_arr.append(gamma * mul1 + R_sum)
        fin_v = np.min(act_arr)
        idx = act_arr.index(fin_v)
        temp = [fin_v, idx]
        send_end.send(temp)
        
    T = 11
    ls = len(all_states)
    V_T = np.zeros(ls)
    Ix_T = np.zeros(ls)
    gamma = 0.99
    stp = 0
    while(stp < T):
        t1 = time.time()
        # print(stp)
        q = 0
        z = 36
        V_Ttemp = np.array([])
        Ix_Ttemp = np.array([])
        while z <= len(all_states):
            jobs = []
            pipe_list = []
            for j in range(q, z):
                recv_end, send_end = multiprocessing.Pipe(False)
                p = Process(target=value_iteration, args = (j, V_T, send_end))
                jobs.append(p)
                pipe_list.append(recv_end)
            for process in jobs:
                process.start()
            for process in jobs:
                process.join()
            temp2 = np.array([x.recv() for x in pipe_list])
            V_Ttemp = np.concatenate((V_Ttemp, temp2[:, 0]))
            Ix_Ttemp = np.concatenate((Ix_Ttemp, temp2[:, 1]))
            q = z
            z+= 36
        V_T = V_Ttemp
        # print('time per itr', time.time() - t1)
        stp+= 1
    # print('value iteration done', e_prob, pkt_prob, weight_prob)
    Ix_T = Ix_Ttemp

    M = 2
    T_horizon = 100000
    tot_dist = 0
    state_slot = [0, 0, 0, 0, 0.1, 0.1,  1, 1]
    wt_next1, wt_next2 = 1, 1
    P1, P2, rho1, rho2 = 0, 0, 0, 0
    for t in range(T_horizon):

        wt1 = np.random.choice([1, 2], p = [weight_prob, (1-weight_prob)])
        wt2 = np.random.choice([1, 2], p = [weight_prob, (1-weight_prob)])
        h1 = np.random.choice([0.1, 1], p = h1_prob)
        h2 = np.random.choice([0.1, 1], p = h1_prob)
        pkt1 = np.random.choice([1, 0], p=[pkt_prob, (1 - pkt_prob)])
        pkt2 = np.random.choice([1, 0], p=[pkt_prob, (1 - pkt_prob)])
        E1 = np.random.choice([1, 0], p=[ e_prob, (1-e_prob)])
        E2 = np.random.choice([1, 0], p=[ e_prob, (1-e_prob)])
        
        if pkt1 == 1:

            tempw1a = wt_next1 * np.exp(-(rmax - (state_slot[2]) ))
            tempw2a = wt1 * 1

            if tempw2a > tempw1a:
                next_bits1 = rmax
                wt_next1 = wt1
            else:
                next_bits1 = state_slot[2]
                
        else:
            next_bits1 = state_slot[2]

        if pkt2 == 1:

            tempw1b = wt_next2 * np.exp(-(rmax - (state_slot[3]) ))
            tempw2b = wt2 * 1

            if tempw2b > tempw1b:
                next_bits2 = rmax
                wt_next2 = wt2
            else:
                state_slot[3]
        else:
            next_bits2 = state_slot[3] 
        # #####################################################################################

        next_B1 = np.minimum(state_slot[0]+ E1, B_max)
        next_B2 = np.minimum(state_slot[1]+ E2, B_max)

        state_slot = [next_B1, next_B2, next_bits1, next_bits2, h1, h2, wt_next1, wt_next2]

        s_idx = all_states.tolist().index(state_slot)
        idx = int(Ix_T[s_idx])
        act = all_actions[idx]
        P1, P2,rho1, rho2 = act[0], act[1], act[2], act[3]
        ################################################################
        dist = wt_next1 * np.exp(-(rmax - (state_slot[2]-rho1) )) + wt_next2 * np.exp(-(rmax - (state_slot[3]-rho2) ))
        state_slot = [state_slot[0]-P1, state_slot[1]-P2, state_slot[2] - rho1, state_slot[3] - rho2, state_slot[4], state_slot[5], state_slot[6], state_slot[7]]
        ########################################################################
        tot_dist+=  dist
    fin_result = tot_dist/(T_horizon*M)
    print('final objective', fin_result)
    print('-------------------------------------')
    # return fin_result
    send_end.send(fin_result)

In [ ]:
# lambda_arr = np.arange(0, 1.1, 0.1)
# jobs = []
# pipe_list = []
# for e_prob in lambda_arr:
#     recv_end, send_end = multiprocessing.Pipe(False)
#     p = Process(target=MDP_two_user, args = (0.5, e_prob, 0.5, 100000, send_end))
#     jobs.append(p)
#     pipe_list.append(recv_end)
# for process in jobs:
#     process.start()
# for process in jobs:
#     process.join()
# final_objective_pkt = np.array([x.recv() for x in pipe_list])

In [ ]:
# lambda_arr = np.arange(0, 1.1, 0.1)
lambda_arr = [0.95]
jobs = []
pipe_list = []
for weight_prob in lambda_arr:
    recv_end, send_end = multiprocessing.Pipe(False)
    p = Process(target=MDP_two_user, args = (0.5, 0.5, weight_prob, 100000, send_end))
    jobs.append(p)
    pipe_list.append(recv_end)
for process in jobs:
    process.start()
for process in jobs:
    process.join()
final_objective_weight = np.array([x.recv() for x in pipe_list])

In [ ]:
# import numpy as np